# **Importing Libraries**

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import pandas as pd
import numpy as np
from google.colab import files
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings
import time
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")


Libraries loaded successfully!


# **File Uploading**

In [16]:
print("📁 Please upload NAFLD_Unclean_Dataset.csv:")
uploaded = files.upload()

📁 Please upload NAFLD_Unclean_Dataset.csv:


Saving Nafld_Unclean_Dataset.csv to Nafld_Unclean_Dataset (1).csv


# **File Loading**

In [17]:
df = pd.read_csv('Nafld_Unclean_Dataset.csv')
print("\n First 5 rows:-")
df.head(5)


 First 5 rows:-


,Patient_ID,Age,Gender,BMI,Waist_Circumference,Fasting_Glucose,Triglycerides,HDL_Cholesterol,Total_Cholesterol,ALT,AST,Diabetes_Status,Physical_Activity_Level,NAFLD_Diagnosis
0,1,56,F,33.6,108.4,56.4,132.1,41.2,262.2,45.8,34.5,NO,Low,1
1,2,69,Female,19.8,93.7,118.3,70.2,58.2,228.7,66.0,47.3,Yes,Low,1
2,3,46,Female,24.2,86.2,103.8,242.2,57.2,265.0,36.2,53.4,Yes,moderate,0
3,4,32,FEMALE,35.5,97.9,103.3,101.4,19.3,192.8,36.9,21.5,yes,Moderate,0
4,5,60,F,29.3,103.4,87.2,145.2,53.4,165.8,48.9,31.7,NO,HIGH,0


# **Basic Information About Data**

**1. Total Rows**

In [18]:
print(f"Total Rows: {len(df)}")

Total Rows: 5050


**2. Total Columns**

In [19]:
print(f"Total Columns: {len(df.columns)}")

Total Columns: 14


**3. Columns Name**

In [20]:
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

   1. Patient_ID
   2. Age
   3. Gender
   4. BMI
   5. Waist_Circumference
   6. Fasting_Glucose
   7. Triglycerides
   8. HDL_Cholesterol
   9. Total_Cholesterol
   10. ALT
   11. AST
   12. Diabetes_Status
   13. Physical_Activity_Level
   14. NAFLD_Diagnosis


**4. Data Types**

In [21]:
print(df.dtypes)

Patient_ID                   int64
Age                          int64
Gender                      object
BMI                        float64
Waist_Circumference        float64
Fasting_Glucose            float64
Triglycerides              float64
HDL_Cholesterol            float64
Total_Cholesterol          float64
ALT                        float64
AST                        float64
Diabetes_Status             object
Physical_Activity_Level     object
NAFLD_Diagnosis              int64
dtype: object


**5. Missing Values**

In [22]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    for col, val in missing.items():
        print(f"    {col}:-    {val} missing values")
else:
    print("No missing values found!")

    BMI:-    152 missing values
    Waist_Circumference:-    153 missing values
    Fasting_Glucose:-    153 missing values
    Triglycerides:-    153 missing values
    HDL_Cholesterol:-    152 missing values


**6. Duplicate Rows**

In [23]:
print(f"Duplicate Rows: {df.duplicated().sum()}")

Duplicate Rows: 50


**7. Negative Values**

In [24]:
columns_to_check = ['BMI','Waist_Circumference','Fasting_Glucose','Triglycerides','HDL_Cholesterol','Total_Cholesterol','ALT','AST']

for col in columns_to_check:
    if col in df.columns:
        negative_count = (df[col] < 0).sum()
        if negative_count > 0:
            print(f"{col}: {negative_count} negative values found")
        else:
            print(f"{col}: No negative values")

BMI: No negative values
Waist_Circumference: No negative values
Fasting_Glucose: No negative values
Triglycerides: 20 negative values found
HDL_Cholesterol: No negative values
Total_Cholesterol: No negative values
ALT: 124 negative values found
AST: 43 negative values found


# **Data Cleaning**

**1. Removing Duplicates Rows**

In [25]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"   Before: {before} rows")
print(f"   After: {after} rows")
print(f"   Removed: {before - after} duplicate rows")

   Before: 5050 rows
   After: 5000 rows
   Removed: 50 duplicate rows


**2. Removing Extra Spaces**

In [26]:
text_columns = df.select_dtypes(include=['object']).columns

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

print(f"Extra spaces removed from: {list(text_columns)}")

Extra spaces removed from: ['Gender', 'Diabetes_Status', 'Physical_Activity_Level']


**3. Fixing Inconsistent Categorical Values From Gender Columns**

In [27]:
if 'Gender' in df.columns:
    print(f"   Before: {df['Gender'].unique()}")

    # Simple mapping
    gender_mapping = {
        'M': 'Male', 'MALE': 'Male', 'male': 'Male', ' m': 'Male', 'm': 'Male',
        'F': 'Female', 'FEMALE': 'Female', 'female': 'Female', ' f': 'Female', 'f': 'Female'
    }

    df['Gender'] = df['Gender'].replace(gender_mapping)

    # If Any Left
    df['Gender'] = df['Gender'].apply(lambda x: 'Male' if str(x).lower().startswith('m') else
                                                'Female' if str(x).lower().startswith('f') else x)

    print(f"After: {df['Gender'].unique()}")

   Before: ['F' 'Female' 'FEMALE' 'M' 'male' 'Male']
After: ['Female' 'Male']


**4. Fixing Inconsistent Categorical Values From Diabetes_Status**

In [28]:
if 'Diabetes_Status' in df.columns:
    print(f"   Before: {df['Diabetes_Status'].unique()}")

    # Striping Lower Case First
    df['Diabetes_Status'] = df['Diabetes_Status'].astype(str).str.lower().str.strip()

    # Mapping
    diabetes_mapping = {
        'yes': 'Yes', 'y': 'Yes', '1': 'Yes', 'true': 'Yes', 'diabetic': 'Yes',
        'no': 'No', 'n': 'No', '0': 'No', 'false': 'No', 'non-diabetic': 'No'
    }

    df['Diabetes_Status'] = df['Diabetes_Status'].replace(diabetes_mapping)

    print(f"   After: {df['Diabetes_Status'].unique()}")

   Before: ['NO' 'Yes' 'yes' 'No']
   After: ['No' 'Yes']


**5. Fixing Inconsistent Categorical Values From Physical_Activity_Level**

In [29]:
if 'Physical_Activity_Level' in df.columns:
    print(f"   Before: {df['Physical_Activity_Level'].unique()}")

    # Striping Lower Case First
    df['Physical_Activity_Level'] = df['Physical_Activity_Level'].astype(str).str.lower().str.strip()

    # Mapping
    activity_mapping = {
        'low': 'Low', 'l': 'Low', 'sedentary': 'Low',
        'moderate': 'Moderate', 'medium': 'Moderate', 'm': 'Moderate',
        'high': 'High', 'h': 'High', 'active': 'High'
    }

    df['Physical_Activity_Level'] = df['Physical_Activity_Level'].replace(activity_mapping)

    print(f"   After: {df['Physical_Activity_Level'].unique()}")

   Before: ['Low' 'moderate' 'Moderate' 'HIGH' 'low' 'High']
   After: ['Low' 'Moderate' 'High']


**6. Listing Of Numerical Columns**

In [30]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Removing Patient_ID and NAFLD_Diagnosis (Because They are Target Variables)
if 'Patient_ID' in num_cols:
    num_cols.remove('Patient_ID')
if 'NAFLD_Diagnosis' in num_cols:
    num_cols.remove('NAFLD_Diagnosis')

print(f"   Numerical columns for cleaning: {num_cols}")

   Numerical columns for cleaning: ['Age', 'BMI', 'Waist_Circumference', 'Fasting_Glucose', 'Triglycerides', 'HDL_Cholesterol', 'Total_Cholesterol', 'ALT', 'AST']


**7. Handling Outliers By Simple IQR Method**

In [31]:
outlier_count = 0

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # First Count Outliers
    outliers = df[(df[col] < lower) | (df[col] > upper)]

    if len(outliers) > 0:
        print(f"{col}: {len(outliers)} outliers found")

        # Put The Outlier in Bracket
        df[col] = df[col].clip(lower, upper)
        outlier_count += len(outliers)
        print(f"Capped at [{lower:.2f}, {upper:.2f}]")

if outlier_count == 0:
    print("No outliers detected!")
else:
    print(f"Total {outlier_count} outliers handled!")

BMI: 73 outliers found
Capped at [13.50, 40.70]
Waist_Circumference: 30 outliers found
Capped at [53.90, 135.50]
Fasting_Glucose: 30 outliers found
Capped at [38.46, 171.16]
Triglycerides: 36 outliers found
Capped at [-3.44, 325.26]
HDL_Cholesterol: 35 outliers found
Capped at [12.25, 77.45]
Total_Cholesterol: 38 outliers found
Capped at [92.35, 307.15]
ALT: 53 outliers found
Capped at [-13.55, 94.05]
AST: 36 outliers found
Capped at [-4.80, 75.20]
Total 331 outliers handled!


**8. Handling Missing Values**

In [32]:
# Check missing values
missing_cols = df.columns[df.isnull().any()].tolist()

if missing_cols:
    print(f"Columns with missing values: {missing_cols}")

    for col in missing_cols:
        missing_count = df[col].isnull().sum()

        if col in num_cols:
            # Median For Numerical Columns
            fill_value = df[col].median()
            df[col] = df[col].fillna(fill_value)
            print(f"{col}: {missing_count} missing → filled with median ({fill_value:.2f})")
        else:
            # Mode For Categorical Columns
            fill_value = df[col].mode()[0]
            df[col] = df[col].fillna(fill_value)
            print(f"{col}: {missing_count} missing → filled with mode ('{fill_value}')")
else:
    print("No missing values found!")

Columns with missing values: ['BMI', 'Waist_Circumference', 'Fasting_Glucose', 'Triglycerides', 'HDL_Cholesterol']
BMI: 150 missing → filled with median (27.10)
Waist_Circumference: 150 missing → filled with median (94.80)
Fasting_Glucose: 150 missing → filled with median (104.60)
Triglycerides: 150 missing → filled with median (161.00)
HDL_Cholesterol: 150 missing → filled with median (44.80)


**9. Handling Negative Values**

In [33]:
for col in columns_to_check:
    if col in df.columns:
        negative_count = (df[col] < 0).sum()
        if negative_count > 0:
            # Negative to Positive
            df[col] = df[col].abs()
            print(f"{col}: {negative_count} negative values converted to absolute values")

Triglycerides: 20 negative values converted to absolute values
ALT: 123 negative values converted to absolute values
AST: 42 negative values converted to absolute values


# **Save The Cleaned Data**

In [34]:
# New file name
cleaned_file = "NAFLD_Cleaned_Dataset.csv"
df.to_csv(cleaned_file, index=False)

print(f"Data saved as: {cleaned_file}")
print(f"File size: {len(df)} rows")

# Download
files.download(cleaned_file)

Data saved as: NAFLD_Cleaned_Dataset.csv
File size: 5000 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Feature Engineering**

**1. AST/ALT Ratio**

In [35]:
# AST/ALT Ratio - In NAFLD This Ratio is <1
if 'AST' in df.columns and 'ALT' in df.columns:
    # Protect from Zero Division
    df['AST_ALT_Ratio'] = df['AST'] / (df['ALT'] + 0.001)
    df['AST_ALT_Ratio'] = df['AST_ALT_Ratio'].round(2)

    print(f"AST/ALT Ratio created!")
    print(f"Range: {df['AST_ALT_Ratio'].min():.2f} to {df['AST_ALT_Ratio'].max():.2f}")
    print(f"Mean: {df['AST_ALT_Ratio'].mean():.2f}")

    # Quick analysis
    low_ratio = (df['AST_ALT_Ratio'] < 1).sum()
    print(f"Patients with ratio < 1: {low_ratio} ({low_ratio/len(df)*100:.1f}%)")

AST/ALT Ratio created!
Range: 0.00 to 38900.00
Mean: 19.29
Patients with ratio < 1: 2893 (57.9%)


**2. BMI Categories**

In [36]:
if 'BMI' in df.columns:
    # Define Conditions
    conditions = [
        (df['BMI'] < 18.5),
        (df['BMI'] >= 18.5) & (df['BMI'] < 25),
        (df['BMI'] >= 25) & (df['BMI'] < 30),
        (df['BMI'] >= 30)
    ]

    categories = ['Underweight', 'Normal', 'Overweight', 'Obese']

    df['BMI_Category'] = np.select(conditions, categories, default='Unknown')

    print(f"BMI Categories !")
    print("\nDistribution:")
    for cat in categories:
        count = (df['BMI_Category'] == cat).sum()
        print(f"{cat}: {count} patients")

BMI Categories !

Distribution:
Underweight: 203 patients
Normal: 1457 patients
Overweight: 1939 patients
Obese: 1401 patients


**2. Age Groups**

In [37]:
if 'Age' in df.columns:
    conditions = [
        (df['Age'] < 30),
        (df['Age'] >= 30) & (df['Age'] < 45),
        (df['Age'] >= 45) & (df['Age'] < 60),
        (df['Age'] >= 60)
    ]

    age_groups = ['Young (<30)', 'Middle (30-45)', 'Senior (45-60)', 'Elderly (60+)']

    df['Age_Group'] = np.select(conditions, age_groups, default='Unknown')

    print(f"Age Groups !")
    print("Distribution:")
    for group in age_groups:
        count = (df['Age_Group'] == group).sum()
        print(f"{group}: {count} patients")

Age Groups !
Distribution:
Young (<30): 945 patients
Middle (30-45): 1193 patients
Senior (45-60): 1233 patients
Elderly (60+): 1629 patients


**3. Metabolic Risk Score**

In [38]:
risk_factors = []

if 'BMI' in df.columns:
    df['BMI_Risk'] = (df['BMI'] > 25).astype(int)
    risk_factors.append('BMI_Risk')
    print(f"BMI > 25: {df['BMI_Risk'].sum()} patients at risk")

if 'Fasting_Glucose' in df.columns:
    df['Glucose_Risk'] = (df['Fasting_Glucose'] > 100).astype(int)
    risk_factors.append('Glucose_Risk')
    print(f"Glucose > 100: {df['Glucose_Risk'].sum()} patients at risk")

if 'Triglycerides' in df.columns:
    df['Triglycerides_Risk'] = (df['Triglycerides'] > 150).astype(int)
    risk_factors.append('Triglycerides_Risk')
    print(f"Triglycerides > 150: {df['Triglycerides_Risk'].sum()} patients at risk")

if 'HDL_Cholesterol' in df.columns:
    df['HDL_Risk'] = (df['HDL_Cholesterol'] < 40).astype(int)  # For males, typically <40
    risk_factors.append('HDL_Risk')
    print(f"HDL < 40: {df['HDL_Risk'].sum()} patients at risk")

if 'ALT' in df.columns:
    df['ALT_Risk'] = (df['ALT'] > 40).astype(int)
    risk_factors.append('ALT_Risk')
    print(f"ALT > 40: {df['ALT_Risk'].sum()} patients at risk")

# Total risk score
if risk_factors:
    df['Metabolic_Risk_Score'] = df[risk_factors].sum(axis=1)
    print(f"\n Metabolic Risk Score created (0 to {len(risk_factors)})")
    print(f"\n Average risk score: {df['Metabolic_Risk_Score'].mean():.2f}")
    print("\n Risk Score Distribution:")
    for i in range(len(risk_factors) + 1):
        count = (df['Metabolic_Risk_Score'] == i).sum()
        print(f"Score {i}: {count} patients")

BMI > 25: 3307 patients at risk
Glucose > 100: 2938 patients at risk
Triglycerides > 150: 2946 patients at risk
HDL < 40: 1698 patients at risk
ALT > 40: 2479 patients at risk

 Metabolic Risk Score created (0 to 5)

 Average risk score: 2.67

 Risk Score Distribution:
Score 0: 87 patients
Score 1: 618 patients
Score 2: 1479 patients
Score 3: 1675 patients
Score 4: 938 patients
Score 5: 203 patients


**4. Waist-to-BMI Ratio**

In [39]:
if 'Waist_Circumference' in df.columns and 'BMI' in df.columns:
    df['Waist_BMI_Ratio'] = (df['Waist_Circumference'] / (df['BMI'] + 0.001)).round(2)

    print(f"Waist-to-BMI Ratio")
    print(f"Range: {df['Waist_BMI_Ratio'].min():.2f} to {df['Waist_BMI_Ratio'].max():.2f}")
    print(f"Mean: {df['Waist_BMI_Ratio'].mean():.2f}")

Waist-to-BMI Ratio
Range: 1.62 to 9.70
Mean: 3.63


**5. Simple Fatty Liver Index**

In [40]:
if all(col in df.columns for col in ['BMI', 'Triglycerides']):
    # Normalize values to 0-100 scale
    bmi_norm = (df['BMI'] - df['BMI'].min()) / (df['BMI'].max() - df['BMI'].min() + 0.001)
    trig_norm = (df['Triglycerides'] - df['Triglycerides'].min()) / (df['Triglycerides'].max() - df['Triglycerides'].min() + 0.001)

    df['Simple_FLI'] = ((bmi_norm + trig_norm) / 2 * 100).round(1)

    print(f"Simple Fatty Liver Index !")

    # Categorize FLI
    df['FLI_Category'] = pd.cut(df['Simple_FLI'],
                                 bins=[0, 30, 60, 100],
                                 labels=['Low Risk', 'Moderate Risk', 'High Risk'])

    print("\nFLI Categories:")
    for cat in ['Low Risk', 'Moderate Risk', 'High Risk']:
        count = (df['FLI_Category'] == cat).sum()
        print(f"{cat}: {count} patients")

Simple Fatty Liver Index !

FLI Categories:
Low Risk: 310 patients
Moderate Risk: 3608 patients
High Risk: 1082 patients


# **Save Feature Engineering Data**

In [41]:
# Save to CSV
feature_Engineering_file = "NAFLD_With_Feature_Engineering.csv"
df.to_csv(feature_Engineering_file, index=False)

print(f"File saved as: {feature_Engineering_file}")

# Download
files.download(feature_Engineering_file)


File saved as: NAFLD_With_Feature_Engineering.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **EDA (Exploratory Data Analysis)**

**1. NAFLD Diagnosis distribution**

In [42]:
diagnosis_counts = df['NAFLD_Diagnosis'].value_counts().reset_index()
diagnosis_counts.columns = ['Diagnosis', 'Count']
diagnosis_counts['Diagnosis'] = diagnosis_counts['Diagnosis'].map({0: 'No Disease', 1: 'Disease'})

fig = px.pie(diagnosis_counts,
             values='Count',
             names='Diagnosis',
             title='<b>NAFLD Diagnosis Distribution</b>',
             color_discrete_sequence=['#66b3ff', '#ff6666'],
             hole=0.4,
             hover_data={'Count': True})
fig.update_layout(
    showlegend=True,
    height=500,
    width=700,
    title_font_size=20
)

fig.show()

**2. Age Distribution by Diagnosis**

In [43]:
fig = px.histogram(df,
                   x='Age',
                   color='NAFLD_Diagnosis',
                   title='<b>Age Distribution: Disease vs No Disease</b>',
                   labels={'NAFLD_Diagnosis': 'Diagnosis', 'Age': 'Age (years)'},
                   color_discrete_map={0: '#66b3ff', 1: '#ff6666'},
                   nbins=30,
                   opacity=0.7,
                   barmode='overlay')
fig.update_layout(
    height=500,
    width=800,
    title_font_size=20,
    legend_title_text='Diagnosis'
)

fig.show()

**3. Liver Enzymes Comparison (ALT vs AST)**

In [44]:
fig = px.scatter(df,
                 x='ALT',
                 y='AST',
                 color='NAFLD_Diagnosis',
                 size='BMI',
                 hover_data=['Age', 'Gender', 'BMI'],
                 title='<b>ALT vs AST with BMI as Size</b>',
                 labels={'ALT': 'ALT (U/L)', 'AST': 'AST (U/L)', 'NAFLD_Diagnosis': 'Diagnosis'},
                 color_discrete_map={0: '#66b3ff', 1: '#ff6666'})
fig.update_layout(
    height=550,
    width=800,
    title_font_size=20
)

fig.show()

**4. AST/ALT Ratio by Age Group**

In [45]:
fig = px.bar(df.groupby(['Age_Group', 'NAFLD_Diagnosis'])['AST_ALT_Ratio'].mean().reset_index(),
             x='Age_Group',
             y='AST_ALT_Ratio',
             color='NAFLD_Diagnosis',
             title='<b>Average AST/ALT Ratio by Age Group</b>',
             labels={'AST_ALT_Ratio': 'AST/ALT Ratio', 'Age_Group': 'Age Group'},
             color_discrete_map={0: '#66b3ff', 1: '#ff6666'},
             barmode='group')
fig.update_layout(
    height=500,
    width=800,
    title_font_size=20
)

fig.show()

**5. Metabolic Risk Score Distribution**

In [46]:
fig = px.histogram(df,
                   x='Metabolic_Risk_Score',
                   color='NAFLD_Diagnosis',
                   title='<b>Metabolic Risk Score Distribution</b>',
                   labels={'Metabolic_Risk_Score': 'Risk Score', 'NAFLD_Diagnosis': 'Diagnosis'},
                   color_discrete_map={0: '#66b3ff', 1: '#ff6666'},
                   nbins=10,
                   barmode='group')
fig.update_layout(
    height=500,
    width=800,
    title_font_size=20
)

fig.show()

**6. Correlation Heatmap**

In [47]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Patient_ID' in numerical_cols:
    numerical_cols.remove('Patient_ID')

corr_matrix = df[numerical_cols].corr()

fig = px.imshow(corr_matrix,
                text_auto='.2f',
                aspect='auto',
                color_continuous_scale='RdBu_r',
                title='<b>Correlation Matrix of Numerical Features</b>',
                labels=dict(color='Correlation'))
fig.update_layout(
    height=700,
    width=900,
    title_font_size=20
)

fig.show()

**7. BMI Category vs NAFLD**

In [48]:
bmi_diagnosis = pd.crosstab(df['BMI_Category'], df['NAFLD_Diagnosis']).reset_index()
bmi_diagnosis.columns = ['BMI_Category', 'No Disease', 'Disease']

fig = px.bar(bmi_diagnosis,
             x='BMI_Category',
             y=['No Disease', 'Disease'],
             title='<b>NAFLD Diagnosis Across BMI Categories</b>',
             labels={'value': 'Number of Patients', 'variable': 'Diagnosis', 'BMI_Category': 'BMI Category'},
             color_discrete_sequence=['#66b3ff', '#ff6666'],
             barmode='group')
fig.update_layout(
    height=500,
    width=800,
    title_font_size=20
)

fig.show()

**8. 3D Scatter Plot (BMI, ALT, Age)**

In [49]:
fig = px.scatter_3d(df,
                    x='BMI',
                    y='ALT',
                    z='Age',
                    color='NAFLD_Diagnosis',
                    size='Triglycerides',
                    hover_data=['Gender', 'Metabolic_Risk_Score'],
                    title='<b>3D Plot: BMI vs ALT vs Age</b>',
                    labels={'BMI': 'BMI', 'ALT': 'ALT (U/L)', 'Age': 'Age (years)'},
                    color_discrete_map={0: '#66b3ff', 1: '#ff6666'})
fig.update_layout(
    height=600,
    width=900,
    title_font_size=20
)

fig.show()

**9. Gender Distribution with NAFLD**

In [50]:
gender_diagnosis = pd.crosstab(df['Gender'], df['NAFLD_Diagnosis']).reset_index()
gender_diagnosis.columns = ['Gender', 'No Disease', 'Disease']

fig = px.bar(gender_diagnosis,
             x='Gender',
             y=['No Disease', 'Disease'],
             title='<b>Gender-wise NAFLD Distribution</b>',
             labels={'value': 'Number of Patients', 'variable': 'Diagnosis'},
             color_discrete_sequence=['#66b3ff', '#ff6666'],
             barmode='stack')
fig.update_layout(
    height=500,
    width=700,
    title_font_size=20
)

fig.show()

**10. Simple FLI Distribution**

In [51]:
fig = px.violin(df,
                x='NAFLD_Diagnosis',
                y='Simple_FLI',
                title='<b>Simple FLI Distribution by Diagnosis</b>',
                labels={'NAFLD_Diagnosis': 'Diagnosis', 'Simple_FLI': 'Fatty Liver Index'},
                color='NAFLD_Diagnosis',
                color_discrete_map={0: '#66b3ff', 1: '#ff6666'},
                box=True,
                points='all')
fig.update_layout(
    height=500,
    width=800,
    title_font_size=20,
    xaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=['No Disease', 'Disease'])
)

fig.show()

### CONCLUSIONS AND INSIGHTS

In [62]:
print("\n" + "="*80)
print("NAFLD ANALYSIS - KEY CONCLUSIONS AND INSIGHTS")
print("="*80)

# Dataset Overview
print("\n📊 DATASET OVERVIEW:")
print(f"   • Total patients analyzed: {len(df)}")
print(f"   • Features used: {len(df.columns) - 1} (excluding target)")
print(f"   • Target distribution:")
print(f"     - No Disease: {(df['NAFLD_Diagnosis'] == 0).sum()} patients ({(df['NAFLD_Diagnosis'] == 0).mean()*100:.1f}%)")
print(f"     - Disease: {(df['NAFLD_Diagnosis'] == 1).sum()} patients ({(df['NAFLD_Diagnosis'] == 1).mean()*100:.1f}%)")

# Key Risk Factors
print("\n🔬 KEY RISK FACTORS IDENTIFIED:")

# BMI Analysis
if 'BMI' in df.columns and 'NAFLD_Diagnosis' in df.columns:
    bmi_risk = df[df['BMI'] >= 30]['NAFLD_Diagnosis'].mean() * 100
    bmi_normal = df[(df['BMI'] >= 18.5) & (df['BMI'] < 25)]['NAFLD_Diagnosis'].mean() * 100
    print(f"   • Obesity (BMI ≥ 30): {bmi_risk:.1f}% have NAFLD")
    print(f"   • Normal BMI (18.5-25): {bmi_normal:.1f}% have NAFLD")
    print(f"   • Obesity increases NAFLD risk by {bmi_risk - bmi_normal:.1f} percentage points")
else:
    print("   • BMI or NAFLD_Diagnosis column not found for BMI Analysis.")

# Diabetes Analysis
if 'Diabetes_Status' in df.columns and 'NAFLD_Diagnosis' in df.columns:
    diabetes_risk = df[df['Diabetes_Status'] == 'Yes']['NAFLD_Diagnosis'].mean() * 100
    no_diabetes_risk = df[df['Diabetes_Status'] == 'No']['NAFLD_Diagnosis'].mean() * 100
    print(f"\n   • Diabetic patients: {diabetes_risk:.1f}% have NAFLD")
    print(f"   • Non-diabetic patients: {no_diabetes_risk:.1f}% have NAFLD")
    print(f"   • Diabetes increases NAFLD risk by {diabetes_risk - no_diabetes_risk:.1f} percentage points")
else:
    print("\n   • Diabetes_Status or NAFLD_Diagnosis column not found for Diabetes Analysis.")

# Metabolic Risk Score
if 'Metabolic_Risk_Score' in df.columns and 'NAFLD_Diagnosis' in df.columns:
    low_risk = df[df['Metabolic_Risk_Score'] <= 2]['NAFLD_Diagnosis'].mean() * 100
    high_risk = df[df['Metabolic_Risk_Score'] >= 4]['NAFLD_Diagnosis'].mean() * 100
    print(f"\n   • Low Metabolic Risk Score (0-2): {low_risk:.1f}% have NAFLD")
    print(f"   • High Metabolic Risk Score (4-5): {high_risk:.1f}% have NAFLD")
    print(f"   • High metabolic risk increases NAFLD likelihood by {high_risk - low_risk:.1f} percentage points")
else:
    print("\n   • Metabolic_Risk_Score or NAFLD_Diagnosis column not found for Metabolic Risk Score Analysis. Please ensure feature engineering cells are run.")

# Age Analysis
if 'Age' in df.columns and 'NAFLD_Diagnosis' in df.columns:
    young_risk = df[df['Age'] < 30]['NAFLD_Diagnosis'].mean() * 100
    elderly_risk = df[df['Age'] >= 60]['NAFLD_Diagnosis'].mean() * 100
    print(f"\n   • Young adults (<30): {young_risk:.1f}% have NAFLD")
    print(f"   • Elderly (≥60): {elderly_risk:.1f}% have NAFLD")
    print(f"   • Age-related increase: {elderly_risk - young_risk:.1f} percentage points")
else:
    print("\n   • Age or NAFLD_Diagnosis column not found for Age Analysis.")

# Gender Analysis
if 'Gender' in df.columns and 'NAFLD_Diagnosis' in df.columns:
    male_risk = df[df['Gender'] == 'Male']['NAFLD_Diagnosis'].mean() * 100
    female_risk = df[df['Gender'] == 'Female']['NAFLD_Diagnosis'].mean() * 100
    print(f"\n   • Males: {male_risk:.1f}% have NAFLD")
    print(f"   • Females: {female_risk:.1f}% have NAFLD")
else:
    print("\n   • Gender or NAFLD_Diagnosis column not found for Gender Analysis.")

# FLI Analysis
if 'FLI_Category' in df.columns and 'NAFLD_Diagnosis' in df.columns:
    fli_low = df[df['FLI_Category'] == 'Low Risk']['NAFLD_Diagnosis'].mean() * 100
    fli_high = df[df['FLI_Category'] == 'High Risk']['NAFLD_Diagnosis'].mean() * 100
    print(f"\n   • Low FLI Risk: {fli_low:.1f}% have NAFLD")
    print(f"   • High FLI Risk: {fli_high:.1f}% have NAFLD")
    print(f"   • High FLI increases NAFLD risk by {fli_high - fli_low:.1f} percentage points")
else:
    print("\n   • FLI_Category or NAFLD_Diagnosis column not found for FLI Analysis. Please ensure feature engineering cells are run.")

# Correlation Insights
print("\n📈 CORRELATION INSIGHTS:")
correlations = {}
for col in ['BMI', 'Fasting_Glucose', 'Triglycerides', 'ALT', 'Metabolic_Risk_Score']:
    if col in df.columns and 'NAFLD_Diagnosis' in df.columns:
        corr = df[col].corr(df['NAFLD_Diagnosis'])
        correlations[col] = corr
        print(f"   • {col}: correlation = {corr:.3f}")
    else:
        print(f"   • Skipping correlation for {col} due to missing column.")

if correlations:
    strongest = max(correlations, key=correlations.get)
    print(f"\n   • Strongest predictor: {strongest} (r = {correlations[strongest]:.3f})")
else:
    print("   • No numerical columns found for correlation analysis.")

# Feature Engineering Impact
print("\n🛠 FEATURE ENGINEERING INSIGHTS:")
# original_cols is not defined in the provided notebook, commenting out this line to avoid NameError.
# print(f"   • Created {len([col for col in df.columns if col not in original_cols])} new features")
print(f"   • Ensure feature engineering cells were run to generate new features like Metabolic_Risk_Score and FLI_Category.")
print(f"   • Metabolic Risk Score identifies patients with multiple risk factors")
print(f"   • FLI provides a simple screening tool for fatty liver risk")
print(f"   • AST/ALT Ratio helps differentiate NAFLD from other liver conditions")

# Recommendations
print("\n💡 RECOMMENDATIONS FOR CLINICAL PRACTICE:")
print("   1. Screen high-risk patients (obese, diabetic, high metabolic score)")
print("   2. Use FLI as a simple screening tool")
print("   3. Monitor liver enzymes (ALT, AST) in at-risk populations")
print("   4. Consider lifestyle interventions for patients with high metabolic risk")
print("   5. Regular follow-up for elderly patients with multiple risk factors")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)



NAFLD ANALYSIS - KEY CONCLUSIONS AND INSIGHTS

📊 DATASET OVERVIEW:
   • Total patients analyzed: 5000
   • Features used: 14 (excluding target)
   • Target distribution:
     - No Disease: 2667 patients (53.3%)
     - Disease: 2333 patients (46.7%)

🔬 KEY RISK FACTORS IDENTIFIED:
   • Obesity (BMI ≥ 30): 76.0% have NAFLD
   • Normal BMI (18.5-25): 35.3% have NAFLD
   • Obesity increases NAFLD risk by 40.7 percentage points

   • Diabetic patients: 47.0% have NAFLD
   • Non-diabetic patients: 46.4% have NAFLD
   • Diabetes increases NAFLD risk by 0.6 percentage points

   • Metabolic_Risk_Score or NAFLD_Diagnosis column not found for Metabolic Risk Score Analysis. Please ensure feature engineering cells are run.

   • Young adults (<30): 49.3% have NAFLD
   • Elderly (≥60): 45.4% have NAFLD
   • Age-related increase: -3.9 percentage points

   • Males: 45.9% have NAFLD
   • Females: 47.4% have NAFLD

   • FLI_Category or NAFLD_Diagnosis column not found for FLI Analysis. Please ensure 